# Stage 09 — Homework Starter Notebook

In the lecture, we learned how to create engineered features. Now it’s your turn to apply those ideas to your own project data.

In [12]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install matplotlib

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1] / "project"

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd() / "project"

sys.path.insert(0, str(PROJECT_ROOT))

data_path = PROJECT_ROOT / "data" / "processed" / "sp500_sector_prices_clean.csv"

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["sector", "date"]).reset_index(drop=True)

df.head()

,date,ticker,sector,close
0,2021-08-30 13:30:00,XLC,Communication Services,85.300003
1,2021-08-31 13:30:00,XLC,Communication Services,85.620003
2,2021-09-01 13:30:00,XLC,Communication Services,86.050003
3,2021-09-02 13:30:00,XLC,Communication Services,85.470001
4,2021-09-03 13:30:00,XLC,Communication Services,85.470001


## Engineered features — one is worked below; add at least two more

This project creates the following features:

1. Five-day momentum to measure recent sector performance.
2. Twenty-day rolling mean return to measure persistent short-term performance.
3. One-hot encoded sector indicators to represent sector membership numerically.

In [14]:
# Feature 1: five-day momentum

df["momentum_5d"] = (
    df.groupby("sector")["close"].pct_change(5)
)

df[["date", "sector", "close", "momentum_5d"]].head(10)

,date,sector,close,momentum_5d
0,2021-08-30 13:30:00,Communication Services,85.300003,NaN
1,2021-08-31 13:30:00,Communication Services,85.620003,NaN
2,2021-09-01 13:30:00,Communication Services,86.050003,NaN
3,2021-09-02 13:30:00,Communication Services,85.470001,NaN
4,2021-09-03 13:30:00,Communication Services,85.470001,NaN
5,2021-09-07 13:30:00,Communication Services,85.669998,0.004338
6,2021-09-08 13:30:00,Communication Services,85.230003,-0.004555
7,2021-09-09 13:30:00,Communication Services,84.809998,-0.014410
8,2021-09-10 13:30:00,Communication Services,84.260002,-0.014157
9,2021-09-13 13:30:00,Communication Services,84.470001,-0.011700


### Rationale for Feature 1
Explain why this feature may help a model. Reference your EDA.

The five-day momentum feature measures the recent price performance of each sector.

This feature supports the project goal of comparing sector performance. It uses the price from five trading days earlier and is calculated separately for each sector. The first five observations of each sector contain missing values because there is not enough historical data.

In [15]:
# Feature 2: twenty-day rolling mean return

df["daily_return"] = (
    df.groupby("sector")["close"].pct_change()
)

# Then calculate the twenty-day rolling mean return
df["rolling_mean_return_20d"] = (
    df.groupby("sector")["daily_return"]
      .transform(lambda s: s.rolling(20, min_periods=20).mean())
)

df[
    ["date", "sector", "daily_return", "rolling_mean_return_20d"]
].head(25)

,date,sector,daily_return,rolling_mean_return_20d
0,2021-08-30 13:30:00,Communication Services,NaN,NaN
1,2021-08-31 13:30:00,Communication Services,0.003751,NaN
2,2021-09-01 13:30:00,Communication Services,0.005022,NaN
3,2021-09-02 13:30:00,Communication Services,-0.006740,NaN
4,2021-09-03 13:30:00,Communication Services,0.000000,NaN
5,2021-09-07 13:30:00,Communication Services,0.002340,NaN
6,2021-09-08 13:30:00,Communication Services,-0.005136,NaN
7,2021-09-09 13:30:00,Communication Services,-0.004928,NaN
8,2021-09-10 13:30:00,Communication Services,-0.006485,NaN
9,2021-09-13 13:30:00,Communication Services,0.002492,NaN


### Rationale for Feature 2
Explain why this feature may help a model. Reference your EDA.

The twenty-day rolling mean return measures the average daily return for each sector during the previous twenty trading days.

This feature reduces short-term noise and helps identify sectors with more persistent positive or negative performance. The first nineteen observations of each sector contain missing values because a complete twenty-day window is not available.

In [16]:
# TODO: Add a third feature. At least one of your three must ENCODE A CATEGORICAL
df_encoded = pd.get_dummies(
    df,
    columns=["sector"],
    prefix="sector",
    dtype=int
)

df_encoded.head()

,date,ticker,close,momentum_5d,daily_return,rolling_mean_return_20d,sector_Communication Services,sector_Consumer Discretionary,sector_Consumer Staples,sector_Energy,sector_Financials,sector_Health Care,sector_Industrials,sector_Information Technology,sector_Materials,sector_Real Estate,sector_Utilities
0,2021-08-30 13:30:00,XLC,85.300003,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0
1,2021-08-31 13:30:00,XLC,85.620003,NaN,0.003751,NaN,1,0,0,0,0,0,0,0,0,0,0
2,2021-09-01 13:30:00,XLC,86.050003,NaN,0.005022,NaN,1,0,0,0,0,0,0,0,0,0,0
3,2021-09-02 13:30:00,XLC,85.470001,NaN,-0.006740,NaN,1,0,0,0,0,0,0,0,0,0,0
4,2021-09-03 13:30:00,XLC,85.470001,NaN,0.000000,NaN,1,0,0,0,0,0,0,0,0,0,0


In [17]:
df[
    ["momentum_5d", "rolling_mean_return_20d", "daily_return"]
].corr()

,momentum_5d,rolling_mean_return_20d,daily_return
momentum_5d,1.000000,0.485702,0.429954
rolling_mean_return_20d,0.485702,1.000000,0.217205
daily_return,0.429954,0.217205,1.000000


In [18]:
# Feature 3: one-hot encode the sector category

df_features = pd.get_dummies(
    df,
    columns=["sector"],
    prefix="sector",
    dtype=int
)

sector_columns = [
    column for column in df_features.columns
    if column.startswith("sector_")
]

df_features[sector_columns].head()

,sector_Communication Services,sector_Consumer Discretionary,sector_Consumer Staples,sector_Energy,sector_Financials,sector_Health Care,sector_Industrials,sector_Information Technology,sector_Materials,sector_Real Estate,sector_Utilities
0,1,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0


### Rationale for Feature 3
Explain why this feature may help a model. Reference your EDA. If this is your
categorical encoding, say why you chose that encoding over the other two.

I use one-hot encoding for the categorical variable `sector`.

Sector is important because the project compares different S&P 500 sectors. One-hot encoding allows a model to treat each sector separately without creating an artificial ranking.

I chose one-hot encoding instead of label encoding because label encoding would incorrectly imply that one sector is greater than another. Frequency encoding is less useful because the sectors have similar numbers of observations.

In [19]:
df_features[sector_columns].sum()

sector_Communication Services    1255
sector_Consumer Discretionary    1255
sector_Consumer Staples          1255
sector_Energy                    1255
sector_Financials                1255
sector_Health Care               1255
sector_Industrials               1255
sector_Information Technology    1255
sector_Materials                 1255
sector_Real Estate               1255
sector_Utilities                 1255
dtype: int64

## Feature Correlation Summary

The following table compares the engineered features with the target variable.
Correlation is used as a simple screening tool rather than proof of causality.

In [20]:
feature_columns = [
    "daily_return",
    "momentum_5d",
    "rolling_mean_return_20d",
]

df[feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
daily_return,13794.0,0.000372,0.012673,-0.091999,-0.006418,0.000706,0.007392,0.134257
momentum_5d,13750.0,0.001860,0.027767,-0.186808,-0.013790,0.002611,0.018174,0.141542
rolling_mean_return_20d,13585.0,0.000389,0.002664,-0.012774,-0.001233,0.000477,0.002075,0.012299


In [21]:
from src.features import add_return_features

project_df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "sp500_sector_data.csv"
)

project_df["date"] = pd.to_datetime(project_df["date"])

project_df = add_return_features(project_df)

project_df["momentum_5d"] = (
    project_df.groupby("sector")["close"].pct_change(5)
)

project_df["rolling_mean_return_20d"] = (
    project_df.groupby("sector")["daily_return"]
              .transform(lambda s: s.rolling(20, min_periods=20).mean())
)

project_df.head()

,date,ticker,sector,close,daily_return,cumulative_return,rolling_volatility_5d,rolling_volatility_20d,month,momentum_5d,rolling_mean_return_20d
0,2021-08-30 13:30:00,XLC,Communication Services,85.300003,NaN,0.000000,NaN,NaN,8,NaN,NaN
1,2021-08-31 13:30:00,XLC,Communication Services,85.620003,0.003751,0.003751,NaN,NaN,8,NaN,NaN
2,2021-09-01 13:30:00,XLC,Communication Services,86.050003,0.005022,0.008792,NaN,NaN,9,NaN,NaN
3,2021-09-02 13:30:00,XLC,Communication Services,85.470001,-0.006740,0.001993,NaN,NaN,9,NaN,NaN
4,2021-09-03 13:30:00,XLC,Communication Services,85.470001,0.000000,0.001993,NaN,NaN,9,NaN,NaN


In [22]:
output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sp500_sector_features_stage09.csv"
)

df_features.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", df_features.shape)

Saved: c:\Users\qwqqqyf\bootcamp_yufei_qin\project\data\processed\sp500_sector_features_stage09.csv
Shape: (13805, 17)
